# Mobile Money Fraud Detection

## AI Engineering Challenge - Phase 1: Data Science

**Candidate Name:** [Western Onzere]
**Date:** [March 17, 2026]

---

## Table of Contents

1. [Setup & Data Loading](#1-setup--data-loading)
2. [Exploratory Data Analysis](#2-exploratory-data-analysis)
   - 2.1 Basic Statistics
   - 2.2 Target Variable Analysis
   - 2.3 Temporal Analysis
   - 2.4 Transaction Type Analysis
   - 2.5 Amount Analysis
   - 2.6 Geographic Analysis
   - 2.7 User Behavior Analysis
3. [Feature Engineering](#3-feature-engineering)
   - 3.1 Temporal Features
   - 3.2 Transaction Velocity Features
   - 3.3 Amount-Based Features
   - 3.4 Behavioral Features
   - 3.5 Device & Location Features
4. [Data Preprocessing](#4-data-preprocessing)
5. [Model Development](#5-model-development)
   - 5.1 Baseline Model
   - 5.2 Model Comparison
   - 5.3 Hyperparameter Tuning
   - 5.4 Final Model Evaluation
6. [Model Interpretation](#6-model-interpretation)
7. [Conclusions & Next Steps](#7-conclusions--next-steps)
8. [Test Set Predictions](#8-test-set-predictions) ⭐ REQUIRED

---

## 1. Setup & Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Set display options
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
# Load the data
# --- 1.3 Load the Dataset ---

FILE_PATH = '/home/western/Desktop/machine-learning/financial-fraud/data/transactions.csv'

df = pd.read_csv(FILE_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")


In [ ]:
# Basic data inspection
# TODO: Display basic info about the dataset
df.info()


In [ ]:
df.head()

In [ ]:
df.shape


## Missing values


In [ ]:
missing_values = df.isnull().sum()
if missing_values.sum() > 0:
    print("\n--- Missing Values Found ---")
    print(missing_values[missing_values > 0])
else:
    print("\nNo missing values found.")


In [ ]:
# Check for duplicates
duplicates = df.duplicated(subset=['transaction_id']).sum()
print(f"\nDuplicate transaction_ids: {duplicates}")



In [ ]:
# Look at categorical variety
categorical_cols = df.select_dtypes(include=['object']).columns
print("\n--- Unique counts for Categorical Columns ---")
for col in categorical_cols:
    if col != 'transaction_id':  # Skip ID to save time
        print(f"{col}: {df[col].nunique()} unique values")

---

## 2. Exploratory Data Analysis

### 2.1 Basic Statistics

In [ ]:

# =============================================================================
# STEP 2: EXPLORATORY DATA ANALYSIS
# =============================================================================

# --- 2.1 Basic Statistics ---

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total transactions : {len(df):,}")
print(f"Time period        : {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Unique senders     : {df['sender_id'].nunique():,}")
print(f"Unique receivers   : {df['receiver_id'].nunique():,}")
print(f"Unique devices     : {df['device_id'].nunique():,}")
print(f"Transaction types  : {df['transaction_type'].unique()}")
print(f"Fraud cases        : {df['is_fraud'].sum():,} ({df['is_fraud'].mean()*100:.2f}%)")

In [ ]:
print("\n" + "=" * 60)
print("NUMERIC COLUMN STATISTICS")
print("=" * 60)

numeric_cols = ['amount', 'sender_balance_before', 'sender_balance_after',
                'receiver_balance_before', 'receiver_balance_after']

stats = df[numeric_cols].describe().T
stats['skewness'] = df[numeric_cols].skew()
# coefficient of variation
stats['cv'] = stats['std'] / stats['mean']

print(stats.round(2))

In [ ]:
# --- Narrative insight block ---

amount_skew = df['amount'].skew()
median_amount = df['amount'].median()
mean_amount = df['amount'].mean()
max_amount = df['amount'].max()
fraud_rate = df['is_fraud'].mean() * 100

print(f"""
1. CLASS IMBALANCE
   - Only {fraud_rate:.2f}% of transactions are fraudulent.
   - This is expected in fraud detection but means accuracy alone
     is a misleading metric — we will optimize for AUC-ROC and
     Precision-Recall instead.

2. TRANSACTION AMOUNTS (KES)
   - Mean: KES {mean_amount:,.0f} | Median: KES {median_amount:,.0f}
   - The mean is significantly higher than the median, indicating
     a right-skewed distribution — a few very large transactions
     pull the average up. (Skewness = {amount_skew:.2f})
   - Max transaction: KES {max_amount:,.0f}
   - We will apply log-transformation to amount during feature
     engineering to reduce the effect of outliers.

3. BALANCE COLUMNS
   - We have sender & receiver balances before and after each
     transaction. These allow us to compute:
       * balance change (actual vs expected)
       * whether a sender drained their account completely
       * receiver balance anomalies
   - These derived features are likely to be strong fraud signals.

4. GEOGRAPHIC COVERAGE
   - Transactions include lat/lon coordinates, enabling spatial
     clustering analysis — fraudulent activity may concentrate
     in specific regions or show unusual location jumps.
""")

In [ ]:
# --- Parse timestamp now so all later steps can use it ---
df['timestamp'] = pd.to_datetime(df['timestamp'])
print("Timestamp parsed to datetime")
print(f"   Date range: {df['timestamp'].dt.date.min()} → {df['timestamp'].dt.date.max()}")
print(f"   Span: {(df['timestamp'].max() - df['timestamp'].min()).days} days")

### 2.2 Target Variable Analysis

In [ ]:
# TODO: Analyze the distribution of fraud vs legitimate transactions

# --- Class Distribution ---
fraud_counts = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].value_counts(normalize=True) * 100

print("=" * 60)
print("FRAUD vs LEGITIMATE DISTRIBUTION")
print("=" * 60)
print(f"Legitimate (0) : {fraud_counts[0]:,}  ({fraud_pct[0]:.2f}%)")
print(f"Fraudulent (1) : {fraud_counts[1]:,}  ({fraud_pct[1]:.2f}%)")
print(f"Imbalance ratio: {fraud_counts[0]/fraud_counts[1]:.0f}:1  (legitimate:fraud)")

In [ ]:
# --- Plot 1: Bar chart showing count of legitimate vs fraud transactions ---

plt.figure(figsize=(6, 5))
bars = plt.bar(['Legitimate', 'Fraud'], fraud_counts.values, color=['#2ecc71', '#e74c3c'])

# Add labels
for i, bar in enumerate(bars):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + (yval*0.02),
             f'{fraud_counts.values[i]:,}\n({fraud_pct.values[i]:.1f}%)',
             ha='center', fontweight='bold')

plt.title('Distribution of Transactions')
plt.ylabel('Count')
plt.ylim(0, max(fraud_counts.values) * 1.2) # Add headroom for labels

# Save directly to current directory
plt.savefig('eda_class_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:

# --- Plot: Histogram comparing transaction amount distribution (log scale) ---
fig, ax = plt.subplots(figsize=(7, 5))

# Plotting for Legitimate (0) and Fraud (1)
for label, color, name in zip([0, 1], ['#2ecc71', '#e74c3c'], ['Legitimate', 'Fraud']):
    # Filter data and apply log transformation
    data = np.log1p(df[df['is_fraud'] == label]['amount'])

    ax.hist(
        data,
        bins=50,
        alpha=0.6,
        color=color,
        label=name,
        edgecolor='none',
        density=True # Use density to compare distributions even if imbalance is high
    )

ax.set_title('Log(1 + Amount) Distribution by Class', fontweight='bold')
ax.set_xlabel('Log(1 + Amount)')
ax.set_ylabel('Density')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('outputs/eda_amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Insight: High-Value Anomaly Detection potential.
- While legitimate transactions follow a unimodal distribution centered at lower amounts, fraudulent transactions are multimodal. Specifically, fraud shows significant spikes at high-value thresholds (Log 11+) where legitimate activity is nearly non-existent.
### Action for Feature Engineering:
- I should create features that flag "high-amount" outliers or bin these specific high-density fraud ranges, as they provide a very clear signal for the model.


In [ ]:
# --- Amount statistics split by class ---
print("\n" + "=" * 60)
print("TRANSACTION AMOUNT STATS BY CLASS (KES)")
print("=" * 60)
amount_by_class = df.groupby('is_fraud')['amount'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).rename(index={0: 'Legitimate', 1: 'Fraud'})
amount_by_class.columns = ['Count', 'Mean', 'Median', 'Std Dev', 'Min', 'Max']
print(amount_by_class.round(0))

In [ ]:
# --- Narrative ---
legit_median = df[df['is_fraud']==0]['amount'].median()
fraud_median = df[df['is_fraud']==1]['amount'].median()
ratio = fraud_counts[0] / fraud_counts[1]

print(f"""
{'='*60}
KEY OBSERVATIONS — TARGET VARIABLE
{'='*60}

1. SEVERE CLASS IMBALANCE
   - The dataset is highly imbalanced at {ratio:.0f}:1 (legitimate:fraud).
   - A naive model predicting all transactions as legitimate
     would achieve 96% accuracy — yet catch zero fraud cases.
   - Strategy: We will use class_weight='balanced' in tree-based
     models and apply SMOTE during preprocessing to synthetically
     oversample the minority class.

2. FRAUD TRANSACTIONS INVOLVE HIGHER AMOUNTS
   - Median amount — Legitimate: KES {legit_median:,.0f} | Fraud: KES {fraud_median:,.0f}
   - Fraudsters tend to target larger transactions, likely to
     maximise gain per attempt before detection.
   - This confirms that 'amount' will be an important feature,
     especially after log-transformation to handle the skew of {9.11}.

3. IMPLICATIONS FOR MODELLING
   - Primary metric  : AUC-ROC (handles imbalance well)
   - Secondary metric: Precision-Recall AUC (focuses on minority class)
   - We will NOT use raw accuracy as a success measure.
   - A high Recall is preferred over Precision in this domain —
     missing a fraudulent transaction (false negative) is more
     costly than a false alarm (false positive).
""")

### 2.3 Temporal Analysis

In [ ]:
# TODO: Analyze transaction patterns over time (hourly, daily, weekly)
# Extract time components — used across all temporal plots below
df['hour']       = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek          # 0=Monday, 6=Sunday
df['day_name']   = df['timestamp'].dt.day_name()
df['week']       = df['timestamp'].dt.isocalendar().week.astype(int)
df['month']      = df['timestamp'].dt.month
df['date']       = df['timestamp'].dt.date

print("Temporal features extracted")
print(f"   Hours range    : {df['hour'].min()}h – {df['hour'].max()}h")
print(f"   Days covered   : {df['day_name'].unique()}")
print(f"   Weeks covered  : Week {df['week'].min()} – Week {df['week'].max()}")

In [ ]:
# --- 2.3.1 Hourly Transaction Patterns ---

hourly = df.groupby('hour').agg(
    total=('is_fraud', 'count'),
    fraud=('is_fraud', 'sum')
).assign(fraud_rate=lambda x: (x['fraud'] / x['total']) * 100)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle('Hourly Transaction Patterns', fontsize=14, fontweight='bold')

# Volume by hour
axes[0].bar(hourly.index, hourly['total'], color='steelblue',
            alpha=0.7, label='Legitimate')
axes[0].bar(hourly.index, hourly['fraud'], color='#e74c3c',
            alpha=0.9, label='Fraud')
axes[0].set_ylabel('Number of Transactions')
axes[0].set_title('Transaction Volume by Hour of Day')
axes[0].set_xticks(range(0, 24))
axes[0].legend()

# Fraud rate by hour
axes[1].plot(hourly.index, hourly['fraud_rate'],
             color='#e74c3c', marker='o', linewidth=2, markersize=5)
axes[1].fill_between(hourly.index, hourly['fraud_rate'], alpha=0.15, color='#e74c3c')
axes[1].axhline(y=df['is_fraud'].mean() * 100, color='gray',
                linestyle='--', linewidth=1.2, label='Overall fraud rate (4%)')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Hour of Day')
axes[1].set_xticks(range(0, 24))
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/eda_2_3_hourly.png', dpi=150, bbox_inches='tight')
plt.show()

# Insight
peak_fraud_hour = hourly['fraud_rate'].idxmax()
peak_volume_hour = hourly['total'].idxmax()

print(f"""
HOURLY INSIGHTS
---------------
- Peak transaction volume : {peak_volume_hour}:00h
  (people transacting most during business hours)
- Highest fraud rate      : {peak_fraud_hour}:00h
  ({hourly.loc[peak_fraud_hour, 'fraud_rate']:.1f}% of transactions at that hour are fraud)
- Late-night hours (00h–05h) show disproportionately high fraud rates
  relative to transaction volume — a classic pattern where fraudsters
  operate when monitoring and user awareness are lowest.
- This suggests 'is_night' (00h–05h) will be a strong binary feature
  in our model.
""")

In [ ]:
# --- 2.3.2 Daily Transaction Patterns ---

daily = df.groupby('date').agg(
    total=('is_fraud', 'count'),
    fraud=('is_fraud', 'sum')
).assign(fraud_rate=lambda x: (x['fraud'] / x['total']) * 100)

fig, axes = plt.subplots(2, 1, figsize=(16, 9))
fig.suptitle('Daily Transaction Patterns (Jan–Jun 2024)', fontsize=14, fontweight='bold')

# Daily volume over time
axes[0].plot(daily.index, daily['total'], color='steelblue',
             linewidth=1.2, alpha=0.8)
axes[0].fill_between(daily.index, daily['total'], alpha=0.15, color='steelblue')
axes[0].set_ylabel('Number of Transactions')
axes[0].set_title('Daily Transaction Volume')
axes[0].tick_params(axis='x', rotation=45)

# Daily fraud count over time
axes[1].plot(daily.index, daily['fraud'], color='#e74c3c',
             linewidth=1.2, alpha=0.8)
axes[1].fill_between(daily.index, daily['fraud'], alpha=0.15, color='#e74c3c')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Fraud Cases')
axes[1].set_title('Daily Fraud Cases Over Time')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('outputs/eda_2_3_daily.png', dpi=150, bbox_inches='tight')
plt.show()

# Insight
peak_fraud_day = daily['fraud'].idxmax()
peak_volume_day = daily['total'].idxmax()

print(f"""
DAILY INSIGHTS
--------------
- Dataset spans 181 days (Jan 1 – Jun 30, 2024).
- Peak transaction day : {peak_volume_day} ({daily.loc[peak_volume_day, 'total']:,} transactions)
- Highest fraud day    : {peak_fraud_day} ({daily.loc[peak_fraud_day, 'fraud']} fraud cases)
- Looking for spikes in fraud count relative to volume helps identify
  potential coordinated fraud campaigns — sudden bursts above the
  rolling average are a red flag worth investigating.
""")

In [ ]:
# --- 2.3.3 Day-of-Week Patterns ---

dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

dow = df.groupby('day_name').agg(
    total=('is_fraud', 'count'),
    fraud=('is_fraud', 'sum')
).assign(fraud_rate=lambda x: (x['fraud'] / x['total']) * 100)
dow = dow.reindex(dow_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Day-of-Week Transaction Patterns', fontsize=14, fontweight='bold')

# Volume by day of week
axes[0].bar(dow.index, dow['total'], color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_ylabel('Number of Transactions')
axes[0].set_title('Transaction Volume by Day of Week')
axes[0].tick_params(axis='x', rotation=30)

# Fraud rate by day of week
colors = ['#e74c3c' if r == dow['fraud_rate'].max() else '#f1948a' for r in dow['fraud_rate']]
axes[1].bar(dow.index, dow['fraud_rate'], color=colors, alpha=0.85, edgecolor='white')
axes[1].axhline(y=df['is_fraud'].mean() * 100, color='gray',
                linestyle='--', linewidth=1.2, label='Overall fraud rate (4%)')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Day of Week')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/eda_2_3_day_of_week.png', dpi=150, bbox_inches='tight')
plt.show()

# Insight
peak_fraud_dow = dow['fraud_rate'].idxmax()
peak_volume_dow = dow['total'].idxmax()

print(f"""
DAY-OF-WEEK INSIGHTS
--------------------
- Highest transaction volume : {peak_volume_dow}
- Highest fraud rate         : {peak_fraud_dow}
  ({dow.loc[peak_fraud_dow, 'fraud_rate']:.1f}% fraud rate vs 4.0% overall)
- Weekend transactions typically drop in volume on M-Pesa-style platforms
  as business payments decline — but fraud rate may remain elevated,
  since fewer active users means anomalies are harder to catch in real time.
- 'day_of_week' and 'is_weekend' will be engineered as features in Step 3.
""")

### 2.4 Transaction Type Analysis

In [ ]:
# --- 2.4 Transaction Type Analysis — Base Aggregation ---

type_stats = df.groupby('transaction_type').agg(
    total=('is_fraud', 'count'),
    fraud=('is_fraud', 'sum'),
    avg_amount=('amount', 'mean'),
    median_amount=('amount', 'median')
).assign(
    fraud_rate=lambda x: (x['fraud'] / x['total']) * 100,
    pct_of_volume=lambda x: (x['total'] / len(df)) * 100
).sort_values('fraud_rate', ascending=False)

print("=" * 65)
print("FRAUD STATS BY TRANSACTION TYPE")
print("=" * 65)
print(type_stats.round(2))

In [ ]:
# TODO: Analyze fraud rates across different transaction types


In [ ]:
# --- 2.4.1 Fraud Rate vs Volume by Transaction Type ---

fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#e74c3c' if r == type_stats['fraud_rate'].max()
          else 'steelblue' for r in type_stats['fraud_rate']]

bars = ax.barh(type_stats.index, type_stats['fraud_rate'],
               color=colors, alpha=0.8, edgecolor='white', height=0.5)

# Annotate with volume share
for bar, (_, row) in zip(bars, type_stats.iterrows()):
    ax.text(bar.get_width() + 0.05,
            bar.get_y() + bar.get_height() / 2,
            f"{row['fraud_rate']:.1f}%  |  {row['pct_of_volume']:.1f}% of volume",
            va='center', fontsize=10)

ax.axvline(x=4.0, color='gray', linestyle='--', linewidth=1.2, label='Overall fraud rate (4.0%)')
ax.set_xlabel('Fraud Rate (%)')
ax.set_title('Fraud Rate by Transaction Type\n(annotated with % share of total volume)',
             fontweight='bold')
ax.set_xlim(0, type_stats['fraud_rate'].max() + 1.5)
ax.legend()

plt.tight_layout()
plt.savefig('outputs/eda_2_4_fraud_rate_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

highest_type = type_stats['fraud_rate'].idxmax()
lowest_type  = type_stats['fraud_rate'].idxmin()

print(f"""
TRANSACTION TYPE — FRAUD RATE INSIGHTS
---------------------------------------
- Fraud rate is remarkably uniform across all five transaction types,
  ranging narrowly from {type_stats['fraud_rate'].min():.1f}% ({lowest_type})
  to {type_stats['fraud_rate'].max():.1f}% ({highest_type}).

- This tells us fraudsters are not concentrating on one channel —
  they operate across the entire platform, which makes transaction_type
  alone a weak predictor. Its value will come in combination with
  amount and time features.

- deposit ({type_stats.loc['deposit', 'fraud_rate']:.1f}%) and
  withdraw ({type_stats.loc['withdraw', 'fraud_rate']:.1f}%) lead on fraud rate
  despite making up only {type_stats.loc['deposit', 'pct_of_volume']:.1f}% and
  {type_stats.loc['withdraw', 'pct_of_volume']:.1f}% of total volume respectively.
  These are higher-value transaction types where a single fraudulent
  transaction causes more financial damage.

- send_money dominates volume at {type_stats.loc['send_money', 'pct_of_volume']:.1f}% of all
  transactions, meaning it contributes the most fraud cases in absolute
  numbers ({type_stats.loc['send_money', 'fraud']:,}) — even at a moderate fraud rate.
""")

In [ ]:
# --- 2.4.2 Median Transaction Amount — Fraud vs Legitimate by Type ---

amount_by_type = df.groupby(['transaction_type', 'is_fraud'])['amount'].median().unstack()
amount_by_type.columns = ['Legitimate', 'Fraud']
amount_by_type = amount_by_type.sort_values('Fraud', ascending=False)

x     = np.arange(len(amount_by_type))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

bars_l = ax.bar(x - width/2, amount_by_type['Legitimate'],
                width, label='Legitimate', color='steelblue', alpha=0.75, edgecolor='white')
bars_f = ax.bar(x + width/2, amount_by_type['Fraud'],
                width, label='Fraud', color='#e74c3c', alpha=0.85, edgecolor='white')

# Annotate bars with KES values
for bar in bars_l:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'KES {bar.get_height():,.0f}', ha='center', va='bottom', fontsize=8.5)
for bar in bars_f:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'KES {bar.get_height():,.0f}', ha='center', va='bottom',
            fontsize=8.5, color='#c0392b', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(amount_by_type.index, rotation=15)
ax.set_ylabel('Median Transaction Amount (KES)')
ax.set_title('Median Amount — Fraud vs Legitimate by Transaction Type',
             fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('outputs/eda_2_4_amount_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

print("AMOUNT BY TYPE INSIGHTS")
print("-" * 40)
for tx_type in amount_by_type.index:
    legit  = amount_by_type.loc[tx_type, 'Legitimate']
    fraud  = amount_by_type.loc[tx_type, 'Fraud']
    mult   = fraud / legit
    print(f"  {tx_type:<12}  Legit: KES {legit:>7,.0f}  |  Fraud: KES {fraud:>7,.0f}  |  {mult:.1f}x higher")

print(f"""
- Across every transaction type, fraudulent transactions involve
  significantly higher amounts than legitimate ones.

- The gap is most extreme in deposit and withdraw — the two types
  that already carry the highest fraud rates. Large deposits or
  withdrawals that deviate from a user's historical behaviour
  are a strong fraud signal.

- buy_goods stands out: legitimate median is just KES {amount_by_type.loc['buy_goods', 'Legitimate']:,.0f},
  yet fraud median is KES {amount_by_type.loc['buy_goods', 'Fraud']:,.0f} — a
  {amount_by_type.loc['buy_goods', 'Fraud']/amount_by_type.loc['buy_goods', 'Legitimate']:.1f}x multiple.
  This suggests fraudsters use buy_goods to move large sums
  through a channel that normally handles small everyday purchases —
  making it an anomaly worth flagging specifically.
""")

### 2.5 Amount Analysis

In [ ]:
# --- 2.5.1 Amount Distribution — Fraud vs Legitimate ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transaction Amount Distribution by Class', fontweight='bold')

# Raw amount — just to show the skew problem
axes[0].hist(df[df['is_fraud']==0]['amount'], bins=80, color='steelblue',
             alpha=0.6, label='Legitimate', edgecolor='none')
axes[0].hist(df[df['is_fraud']==1]['amount'], bins=80, color='#e74c3c',
             alpha=0.7, label='Fraud', edgecolor='none')
axes[0].set_xlabel('Amount (KES)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Raw Amount (skewness = 9.11)')
axes[0].legend()

# Log-transformed — the actual useful view
axes[1].hist(np.log1p(df[df['is_fraud']==0]['amount']), bins=60,
             color='steelblue', alpha=0.6, label='Legitimate', edgecolor='none')
axes[1].hist(np.log1p(df[df['is_fraud']==1]['amount']), bins=60,
             color='#e74c3c', alpha=0.7, label='Fraud', edgecolor='none')
axes[1].set_xlabel('log(1 + Amount)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log-Transformed Amount (clearer separation)')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/eda_2_5_amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

legit_median = df[df['is_fraud']==0]['amount'].median()
fraud_median = df[df['is_fraud']==1]['amount'].median()

print(f"""
AMOUNT DISTRIBUTION INSIGHTS
-----------------------------
- Raw amount is nearly unreadable due to extreme right skew (9.11) —
  the vast majority of transactions cluster below KES 10,000 while
  a few outliers stretch to KES 206,029.

- After log-transformation the separation becomes clear: the fraud
  distribution sits visibly to the right of the legitimate one,
  confirming that higher amounts correlate strongly with fraud.

- Legitimate median : KES {legit_median:,.0f}
  Fraud median      : KES {fraud_median:,.0f}  ({fraud_median/legit_median:.1f}x higher)

- log(amount) will be used directly as a model feature. The raw
  amount will be dropped to avoid distorting distance-based
  calculations and gradient updates.
""")

In [ ]:
# --- 2.5.2 Fraud Rate by Amount Bracket ---

df['amount_band'] = pd.cut(
    df['amount'],
    bins=[0, 500, 1000, 2500, 5000, 10000, 25000, 50000, df['amount'].max()],
    labels=['0–500', '500–1K', '1K–2.5K', '2.5K–5K',
            '5K–10K', '10K–25K', '25K–50K', '50K+']
)

band_stats = df.groupby('amount_band', observed=True).agg(
    total=('is_fraud', 'count'),
    fraud=('is_fraud', 'sum')
).assign(fraud_rate=lambda x: (x['fraud'] / x['total']) * 100)

fig, ax1 = plt.subplots(figsize=(12, 5))

ax2 = ax1.twinx()

ax1.bar(band_stats.index, band_stats['total'], color='steelblue',
        alpha=0.4, label='Transaction Volume')
ax2.plot(band_stats.index, band_stats['fraud_rate'], color='#e74c3c',
         marker='o', linewidth=2.5, markersize=7, label='Fraud Rate %')
ax2.axhline(y=4.0, color='gray', linestyle='--', linewidth=1.2, label='Overall (4.0%)')

ax1.set_xlabel('Transaction Amount Band (KES)')
ax1.set_ylabel('Number of Transactions', color='steelblue')
ax2.set_ylabel('Fraud Rate (%)', color='#e74c3c')
ax1.set_title('Fraud Rate vs Transaction Volume by Amount Band', fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('outputs/eda_2_5_fraud_rate_by_band.png', dpi=150, bbox_inches='tight')
plt.show()

peak_band = band_stats['fraud_rate'].idxmax()

print(f"""
AMOUNT BAND INSIGHTS
--------------------
- Fraud rate is not linear — it rises sharply at higher amount bands.
- Peak fraud rate occurs in the '{peak_band}' band
  at {band_stats.loc[peak_band, 'fraud_rate']:.1f}%, well above the 4.0% platform average.

- The bulk of transaction volume sits in the lower bands (KES 0–2,500)
  where fraud rate is below average — this is the normal, everyday
  usage pattern of the platform.

- Beyond KES 10,000 the fraud rate climbs consistently. This threshold
  will be used as a binary feature ('is_high_value') in feature
  engineering — a simple but effective signal for the model.
""")

In [ ]:
# --- 2.5.3 Account Draining — Balance After Transaction ---

# How much of the sender's balance remains after the transaction?
df['balance_remaining_pct'] = (
    df['sender_balance_after'] / (df['sender_balance_before'] + 1)  # +1 avoids division by zero
) * 100

drain_stats = df.groupby('is_fraud')['balance_remaining_pct'].describe()
drain_stats.index = ['Legitimate', 'Fraud']

print("=" * 55)
print("SENDER BALANCE REMAINING AFTER TRANSACTION (%)")
print("=" * 55)
print(drain_stats[['mean', '50%', '25%', 'min']].round(2))

fig, ax = plt.subplots(figsize=(10, 5))

for label, color, name in zip([0, 1], ['steelblue', '#e74c3c'], ['Legitimate', 'Fraud']):
    data = df[df['is_fraud'] == label]['balance_remaining_pct'].clip(0, 100)
    ax.hist(data, bins=60, alpha=0.6, color=color, label=name, edgecolor='none')

ax.axvline(x=10, color='black', linestyle='--', linewidth=1.2, label='10% threshold')
ax.set_xlabel('Sender Balance Remaining After Transaction (%)')
ax.set_ylabel('Frequency')
ax.set_title('Account Draining Pattern — Fraud vs Legitimate', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('outputs/eda_2_5_account_draining.png', dpi=150, bbox_inches='tight')
plt.show()

fraud_drain   = df[df['is_fraud']==1]['balance_remaining_pct'].median()
legit_drain   = df[df['is_fraud']==0]['balance_remaining_pct'].median()
near_zero_fraud = (df[df['is_fraud']==1]['balance_remaining_pct'] < 10).mean() * 100
near_zero_legit = (df[df['is_fraud']==0]['balance_remaining_pct'] < 10).mean() * 100

print(f"""
ACCOUNT DRAINING INSIGHTS
--------------------------
- After a legitimate transaction, the sender retains a median of
  {legit_drain:.1f}% of their balance — consistent with normal partial payments.

- After a fraudulent transaction, the sender retains only {fraud_drain:.1f}%
  — suggesting fraudsters tend to drain accounts as completely
  as possible in a single transaction.

- {near_zero_fraud:.1f}% of fraud transactions leave the sender with less than
  10% of their balance remaining, compared to just {near_zero_legit:.1f}% for
  legitimate transactions.

- This gives us a powerful engineered feature: 'balance_drain_rate'
  (amount / sender_balance_before) — a high ratio flags likely account
  takeover or unauthorised full withdrawal.
""")

### 2.6 Geographic Analysis

In [ ]:
# --- 2.6 Geographic Analysis — Base Aggregation ---

# Bounding box check — confirm coordinates are Kenya-range
print("Lat range :", df['location_lat'].min().round(4), "→", df['location_lat'].max().round(4))
print("Lon range :", df['location_lon'].min().round(4), "→", df['location_lon'].max().round(4))

# Fraud rate by coordinate bins
geo_stats = df.groupby('is_fraud')[['location_lat', 'location_lon']].describe()
print("\n", geo_stats.round(4))

# How spread out are fraud vs legitimate coordinates?
fraud_lat_std    = df[df['is_fraud']==1]['location_lat'].std()
legit_lat_std    = df[df['is_fraud']==0]['location_lat'].std()
fraud_lon_std    = df[df['is_fraud']==1]['location_lon'].std()
legit_lon_std    = df[df['is_fraud']==0]['location_lon'].std()

print(f"""
Coordinate spread (std dev):
  Legitimate — lat: {legit_lat_std:.4f}  lon: {legit_lon_std:.4f}
  Fraud      — lat: {fraud_lat_std:.4f}  lon: {fraud_lon_std:.4f}
""")

In [ ]:
# --- 2.6.1 Geographic Spread — Fraud vs Legitimate ---

# Sample legitimate to avoid overplotting (fraud is already only 2,000 points)
legit_sample = df[df['is_fraud']==0].sample(3000, random_state=42)
fraud_all    = df[df['is_fraud']==1]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Geographic Distribution of Transactions', fontweight='bold')

for ax, data, color, title in zip(
    axes,
    [legit_sample, fraud_all],
    ['steelblue', '#e74c3c'],
    ['Legitimate (sample n=3,000)', 'Fraud (all n=2,000)']
):
    ax.scatter(data['location_lon'], data['location_lat'],
               alpha=0.3, s=8, color=color)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(28, 46)
    ax.set_ylim(-10, 7)

    # Kenya approximate bounding box
    from matplotlib.patches import Rectangle
    kenya_box = Rectangle((34, -5), 8, 10,
                           linewidth=1.5, edgecolor='black',
                           facecolor='none', linestyle='--', label='Kenya approx.')
    ax.add_patch(kenya_box)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('outputs/eda_2_6_geo_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

outside_kenya_fraud = (
    (df[df['is_fraud']==1]['location_lat'] < -5) |
    (df[df['is_fraud']==1]['location_lat'] > 5)  |
    (df[df['is_fraud']==1]['location_lon'] < 34) |
    (df[df['is_fraud']==1]['location_lon'] > 42)
).mean() * 100

outside_kenya_legit = (
    (df[df['is_fraud']==0]['location_lat'] < -5) |
    (df[df['is_fraud']==0]['location_lat'] > 5)  |
    (df[df['is_fraud']==0]['location_lon'] < 34) |
    (df[df['is_fraud']==0]['location_lon'] > 42)
).mean() * 100

print(f"""
GEOGRAPHIC SPREAD INSIGHTS
--------------------------
- Legitimate transactions cluster tightly within Kenya's boundaries
  (lat std: 1.36, lon std: 1.95) — consistent with a platform
  serving a predominantly Kenyan user base.

- Fraud transactions are geographically dispersed far beyond Kenya,
  with coordinate std nearly 2x wider (lat std: 2.94, lon std: 3.23).

- {outside_kenya_fraud:.1f}% of fraud transactions originate outside the
  approximate Kenya bounding box, compared to just {outside_kenya_legit:.1f}%
  of legitimate ones.

- This strongly suggests location spoofing or VPN usage by fraudsters —
  a common pattern in mobile money fraud where bad actors mask their
  true location.

- We will engineer an 'is_outside_kenya' binary feature and also
  compute distance from Nairobi (the platform's likely hub) as a
  continuous fraud signal.
""")

In [ ]:
# --- 2.6.2 Distance from Nairobi as a Fraud Signal ---

# Nairobi coordinates
NAIROBI_LAT = -1.2921
NAIROBI_LON = 36.8219

def haversine_km(lat, lon, ref_lat=NAIROBI_LAT, ref_lon=NAIROBI_LON):
    """Compute distance in km from a reference point using Haversine formula."""
    R = 6371
    lat1, lat2 = np.radians(ref_lat), np.radians(lat)
    dlon = np.radians(lon - ref_lon)
    dlat = np.radians(lat - ref_lat)
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df['dist_from_nairobi'] = haversine_km(df['location_lat'], df['location_lon'])

dist_stats = df.groupby('is_fraud')['dist_from_nairobi'].describe()
dist_stats.index = ['Legitimate', 'Fraud']
print("=" * 55)
print("DISTANCE FROM NAIROBI (km) BY CLASS")
print("=" * 55)
print(dist_stats[['mean', '50%', '75%', 'max']].round(1))

fig, ax = plt.subplots(figsize=(11, 5))

for label, color, name in zip([0, 1], ['steelblue', '#e74c3c'], ['Legitimate', 'Fraud']):
    data = df[df['is_fraud']==label]['dist_from_nairobi']
    ax.hist(data, bins=60, alpha=0.6, color=color,
            label=f'{name} (median: {data.median():.0f} km)', edgecolor='none')

ax.axvline(x=500, color='black', linestyle='--',
           linewidth=1.2, label='500 km threshold')
ax.set_xlabel('Distance from Nairobi (km)')
ax.set_ylabel('Frequency')
ax.set_title('Transaction Distance from Nairobi — Fraud vs Legitimate', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('outputs/eda_2_6_distance_nairobi.png', dpi=150, bbox_inches='tight')
plt.show()

fraud_far   = (df[df['is_fraud']==1]['dist_from_nairobi'] > 500).mean() * 100
legit_far   = (df[df['is_fraud']==0]['dist_from_nairobi'] > 500).mean() * 100
fraud_med   = df[df['is_fraud']==1]['dist_from_nairobi'].median()
legit_med   = df[df['is_fraud']==0]['dist_from_nairobi'].median()

print(f"""
DISTANCE FROM NAIROBI INSIGHTS
-------------------------------
- Legitimate transactions have a median distance of {legit_med:.0f} km
  from Nairobi — most users transact from within or near the city
  and its surrounding counties.

- Fraud transactions show a median distance of {fraud_med:.0f} km —
  and a much longer tail, with {fraud_far:.1f}% originating more than
  500 km away vs {legit_far:.1f}% for legitimate transactions.

- 'dist_from_nairobi' will be included directly as a continuous
  feature. Combined with 'is_outside_kenya', these two geographic
  features give the model a spatial fraud signal that is both
  interpretable and practically meaningful.

- Note: extreme distances may also reflect GPS spoofing rather than
  actual travel — either way, the signal is valid for fraud detection.
""")

### 2.7 User Behavior Analysis

In [ ]:
# TODO: Analyze user-level patterns (device usage, transaction frequency, etc.)


In [ ]:
# --- 2.7 User Behavior Analysis — Base Aggregation ---

# Transaction frequency per sender
sender_stats = df.groupby('sender_id').agg(
    total_tx=('transaction_id', 'count'),
    fraud_tx=('is_fraud', 'sum'),
    unique_devices=('device_id', 'nunique'),
    unique_receivers=('receiver_id', 'nunique'),
    avg_amount=('amount', 'mean')
).assign(fraud_rate=lambda x: (x['fraud_tx'] / x['total_tx']) * 100)

# Device reuse
device_stats = df.groupby('device_id').agg(
    total_tx=('transaction_id', 'count'),
    fraud_tx=('is_fraud', 'sum'),
    unique_senders=('sender_id', 'nunique')
).assign(fraud_rate=lambda x: (x['fraud_tx'] / x['total_tx']) * 100)

print("SENDER BEHAVIOUR SUMMARY")
print("-" * 40)
print(f"Unique senders          : {sender_stats.shape[0]:,}")
print(f"Avg transactions/sender : {sender_stats['total_tx'].mean():.1f}")
print(f"Max transactions/sender : {sender_stats['total_tx'].max()}")
print(f"Senders with >1 device  : {(sender_stats['unique_devices'] > 1).sum():,}")
print(f"Senders with any fraud  : {(sender_stats['fraud_tx'] > 0).sum():,}")

print("\nDEVICE BEHAVIOUR SUMMARY")
print("-" * 40)
print(f"Unique devices           : {device_stats.shape[0]:,}")
print(f"Avg transactions/device  : {device_stats['total_tx'].mean():.1f}")
print(f"Max transactions/device  : {device_stats['total_tx'].max()}")
print(f"Devices used by >1 sender: {(device_stats['unique_senders'] > 1).sum():,}")
print(f"Devices with any fraud   : {(device_stats['fraud_tx'] > 0).sum():,}")

In [ ]:
# --- 2.7.1 Device Switching Behaviour ---

# Merge sender-level device count back to transaction level
df = df.merge(
    sender_stats[['unique_devices', 'total_tx', 'unique_receivers']].rename(columns={
        'unique_devices'  : 'sender_unique_devices',
        'total_tx'        : 'sender_total_tx',
        'unique_receivers': 'sender_unique_receivers'
    }),
    left_on='sender_id', right_index=True, how='left'
)

device_switch = df.groupby('is_fraud')['sender_unique_devices'].value_counts(
    normalize=True
).mul(100).rename('pct').reset_index()
device_switch.columns = ['is_fraud', 'unique_devices', 'pct']
device_switch['class'] = device_switch['is_fraud'].map({0: 'Legitimate', 1: 'Fraud'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Device Usage Patterns by Class', fontweight='bold')

for ax, (label, group), color in zip(
    axes,
    device_switch.groupby('class'),
    ['steelblue', '#e74c3c']
):
    group = group.sort_values('unique_devices').head(8)  # cap at 8 for readability
    ax.bar(group['unique_devices'].astype(str), group['pct'],
           color=color, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Number of Unique Devices Used')
    ax.set_ylabel('% of Transactions')
    ax.set_title(f'{label} Transactions', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/eda_2_7_device_switching.png', dpi=150, bbox_inches='tight')
plt.show()

fraud_multi_device = (df[df['is_fraud']==1]['sender_unique_devices'] > 1).mean() * 100
legit_multi_device = (df[df['is_fraud']==0]['sender_unique_devices'] > 1).mean() * 100
fraud_med_devices  = df[df['is_fraud']==1]['sender_unique_devices'].median()
legit_med_devices  = df[df['is_fraud']==0]['sender_unique_devices'].median()

print(f"""
DEVICE SWITCHING INSIGHTS
--------------------------
- {fraud_multi_device:.1f}% of fraud transactions are made by senders who
  use more than one device, compared to {legit_multi_device:.1f}% for legitimate.

- Fraudsters frequently switch devices — either to evade device-based
  detection rules or because they are operating compromised accounts
  from their own hardware.

- Median unique devices per sender: Fraud={fraud_med_devices:.0f}, Legit={legit_med_devices:.0f}

- 'sender_unique_devices' will be a direct feature. We will also
  engineer a binary 'is_device_switch' flag per transaction by
  checking if the current device matches the sender's most
  frequently used device.
""")

In [ ]:
# --- 2.7.2 Shared Device Analysis ---

# Merge device-level sender count back to transactions
df = df.merge(
    device_stats[['unique_senders']].rename(columns={
        'unique_senders': 'device_unique_senders'
    }),
    left_on='device_id', right_index=True, how='left'
)

shared_stats = df.groupby('device_unique_senders').agg(
    total=('is_fraud', 'count'),
    fraud=('is_fraud', 'sum')
).assign(fraud_rate=lambda x: (x['fraud'] / x['total']) * 100).head(10)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.bar(shared_stats.index.astype(str), shared_stats['total'],
        color='steelblue', alpha=0.4, label='Transaction Volume')
ax2.plot(shared_stats.index.astype(str), shared_stats['fraud_rate'],
         color='#e74c3c', marker='o', linewidth=2.5,
         markersize=7, label='Fraud Rate %')
ax2.axhline(y=4.0, color='gray', linestyle='--',
            linewidth=1.2, label='Overall (4.0%)')

ax1.set_xlabel('Number of Senders Sharing One Device')
ax1.set_ylabel('Transaction Volume', color='steelblue')
ax2.set_ylabel('Fraud Rate (%)', color='#e74c3c')
ax1.set_title('Fraud Rate vs Volume by Device Sharing Level', fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('outputs/eda_2_7_shared_devices.png', dpi=150, bbox_inches='tight')
plt.show()

shared_fraud_rate = df[df['device_unique_senders'] > 1]['is_fraud'].mean() * 100
solo_fraud_rate   = df[df['device_unique_senders'] == 1]['is_fraud'].mean() * 100
max_shared        = df['device_unique_senders'].max()


print(f"""
SHARED DEVICE INSIGHTS — CORRECTED INTERPRETATION
---------------------------------------------------
- Devices used by a single sender have a fraud rate of {solo_fraud_rate:.1f}% —
  actually HIGHER than devices shared across multiple senders ({shared_fraud_rate:.1f}%).

- This is counterintuitive but explainable: fraudsters who operate
  across multiple accounts tend to use dedicated devices per account
  to avoid triggering shared-device detection rules.

- The more actionable signal is therefore NOT device sharing, but
  rather device switching per sender — a sender suddenly transacting
  from a new device they have never used before is a stronger
  individual-level fraud indicator.

- We will drop 'is_shared_device' from our feature plan and instead
  focus on 'sender_unique_devices' and 'is_device_switch' as the
  primary device-based features.
""")

In [ ]:
df.head(10)

In [ ]:
print(df.columns)

### Key EDA Insights

*TODO: Document your key findings from EDA*

1. 
2. 
3.

In [100]:
# =============================================================================
# STEP 3: FEATURE ENGINEERING
# =============================================================================

def add_temporal_features(df):
    """Extract time-based fraud signals from the timestamp column."""
    df = df.copy()

    df['timestamp']   = pd.to_datetime(df['timestamp'])
    df['hour']        = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek   # 0=Monday
    df['month']       = df['timestamp'].dt.month

    # Binary flags — both shown as strong signals in EDA
    df['is_night']   = df['hour'].between(0, 5).astype(int)  # 03h peak fraud
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    return df

In [101]:
def add_amount_features(df):
    """Log-transform amount and flag high-value transactions."""
    df = df.copy()

    # Log-transform — skewness 9.11 makes raw amount unusable
    df['log_amount'] = np.log1p(df['amount'])

    # Threshold from EDA — fraud rate jumps sharply beyond KES 10,000
    df['is_high_value'] = (df['amount'] > 10000).astype(int)

    return df

In [102]:
def add_balance_features(df):
    """
    Capture account draining and balance anomalies.
    These use the before/after balance columns unique to this dataset.
    """
    df = df.copy()

    # How much of sender's balance was spent — high ratio = likely drain
    df['balance_drain_rate'] = (
        df['amount'] / (df['sender_balance_before'] + 1)
    ).clip(0, 1)

    # Expected vs actual balance after transaction
    df['sender_balance_change']   = df['sender_balance_before'] - df['sender_balance_after']
    df['receiver_balance_change'] = df['receiver_balance_after'] - df['receiver_balance_before']

    # Flag transactions where sender balance dropped more than the amount
    # (potential system manipulation or data anomaly worth capturing)
    df['balance_discrepancy'] = (
        (df['sender_balance_change'] - df['amount']).abs() > 1
    ).astype(int)

    return df

In [103]:
def add_geographic_features(df):
    """Distance and boundary features from EDA — near-perfect fraud separators."""
    df = df.copy()

    NAIROBI_LAT = -1.2921
    NAIROBI_LON = 36.8219

    def haversine_km(lat, lon):
        R = 6371
        lat1, lat2 = np.radians(NAIROBI_LAT), np.radians(lat)
        dlat = np.radians(lat - NAIROBI_LAT)
        dlon = np.radians(lon - NAIROBI_LON)
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        return R * 2 * np.arcsin(np.sqrt(a))

    df['dist_from_nairobi'] = haversine_km(df['location_lat'], df['location_lon'])

    # 29.3% fraud vs 0.0% legitimate outside Kenya — near-perfect separator
    df['is_outside_kenya'] = (
        (df['location_lat'] < -5) | (df['location_lat'] > 5) |
        (df['location_lon'] < 34) | (df['location_lon'] > 42)
    ).astype(int)

    return df

In [104]:
def add_behavioural_features(df):
    """
    Sender and device-level behavioural signals.
    Aggregates are computed from the input df — when applied to test
    set, pass training aggregates in to avoid data leakage.
    """
    df = df.copy()

    # --- Sender-level aggregates ---
    sender_agg = df.groupby('sender_id').agg(
        sender_total_tx      =('transaction_id', 'count'),
        sender_avg_amount    =('amount', 'mean'),
        sender_unique_devices=('device_id', 'nunique'),
        sender_unique_recv   =('receiver_id', 'nunique')
    )

    df = df.merge(sender_agg, on='sender_id', how='left')

    # How much does this transaction deviate from sender's average?
    df['amount_vs_sender_avg'] = (
        df['amount'] / (df['sender_avg_amount'] + 1)
    )

    # --- Device switching ---
    # Most frequent device per sender
    primary_device = (
        df.groupby(['sender_id', 'device_id'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .drop_duplicates('sender_id')
        .rename(columns={'device_id': 'primary_device'})
        [['sender_id', 'primary_device']]
    )

    df = df.merge(primary_device, on='sender_id', how='left')
    df['is_device_switch'] = (df['device_id'] != df['primary_device']).astype(int)
    df.drop(columns=['primary_device'], inplace=True)

    # --- Device-level aggregates ---
    device_agg = df.groupby('device_id').agg(
        device_unique_senders=('sender_id', 'nunique')
    )
    df = df.merge(device_agg, on='device_id', how='left')

    return df

In [105]:
def encode_categoricals(df):
    """Label encode transaction_type — consistent across train and test."""
    df = df.copy()

    type_mapping = {
        'send_money': 0,
        'pay_bill'  : 1,
        'buy_goods' : 2,
        'withdraw'  : 3,
        'deposit'   : 4
    }

    df['transaction_type_enc'] = df['transaction_type'].map(type_mapping)

    return df

In [106]:
def engineer_features(df):
    """
    Full feature engineering pipeline.
    Applies all transformations in order and drops columns
    not needed for modelling.
    """
    df = add_temporal_features(df)
    df = add_amount_features(df)
    df = add_balance_features(df)
    df = add_geographic_features(df)
    df = add_behavioural_features(df)
    df = encode_categoricals(df)

    # Columns to drop — identifiers, raw fields superseded by
    # engineered versions, and EDA-only columns
    drop_cols = [
        'timestamp', 'sender_id', 'receiver_id', 'device_id',
        'day_name', 'date', 'week', 'amount_band',
        'balance_remaining_pct',       # superseded by balance_drain_rate
        'transaction_type',            # superseded by transaction_type_enc
        'sender_avg_amount',           # intermediate — captured in amount_vs_sender_avg
    ]

    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    return df


# --- Apply to training data ---
df_featured = engineer_features(df)

print(f"✅ Feature engineering complete")
print(f"   Original columns : {df.shape[1]}")
print(f"   Final columns    : {df_featured.shape[1]}")
print(f"\nFinal feature set:")
print([c for c in df_featured.columns if c != 'is_fraud'])

✅ Feature engineering complete
   Original columns : 27
   Final columns    : 33

Final feature set:
['transaction_id', 'amount', 'sender_balance_before', 'sender_balance_after', 'receiver_balance_before', 'receiver_balance_after', 'location_lat', 'location_lon', 'hour', 'day_of_week', 'month', 'dist_from_nairobi', 'sender_unique_devices_x', 'sender_total_tx_x', 'sender_unique_receivers', 'device_unique_senders_x', 'is_night', 'is_weekend', 'log_amount', 'is_high_value', 'balance_drain_rate', 'sender_balance_change', 'receiver_balance_change', 'balance_discrepancy', 'is_outside_kenya', 'sender_total_tx_y', 'sender_unique_devices_y', 'sender_unique_recv', 'amount_vs_sender_avg', 'is_device_switch', 'device_unique_senders_y', 'transaction_type_enc']


In [107]:
# These columns were created during EDA and are now duplicated

eda_cols_to_drop = [
    'sender_unique_devices',
    'sender_total_tx',
    'sender_unique_receivers',
    'device_unique_senders',
    'dist_from_nairobi',      # also recreated in add_geographic_features
    'balance_remaining_pct',
    'amount_band',
    'hour', 'day_of_week', 'day_name',
    'week', 'month', 'date'
]

df_clean = df.drop(
    columns=[c for c in eda_cols_to_drop if c in df.columns]
)

print("Columns after cleanup:")
print(df_clean.columns.tolist())
print(f"\nShape: {df_clean.shape}")

Columns after cleanup:
['transaction_id', 'timestamp', 'sender_id', 'receiver_id', 'amount', 'transaction_type', 'sender_balance_before', 'sender_balance_after', 'receiver_balance_before', 'receiver_balance_after', 'device_id', 'location_lat', 'location_lon', 'is_fraud']

Shape: (50000, 14)


In [108]:
df_featured = engineer_features(df_clean)

print(f"\n✅ Feature engineering complete")
print(f"   Shape: {df_featured.shape}")
print(f"\nFinal feature set:")
print([c for c in df_featured.columns if c != 'is_fraud'])


✅ Feature engineering complete
   Shape: (50000, 29)

Final feature set:
['transaction_id', 'amount', 'sender_balance_before', 'sender_balance_after', 'receiver_balance_before', 'receiver_balance_after', 'location_lat', 'location_lon', 'hour', 'day_of_week', 'month', 'is_night', 'is_weekend', 'log_amount', 'is_high_value', 'balance_drain_rate', 'sender_balance_change', 'receiver_balance_change', 'balance_discrepancy', 'dist_from_nairobi', 'is_outside_kenya', 'sender_total_tx', 'sender_unique_devices', 'sender_unique_recv', 'amount_vs_sender_avg', 'is_device_switch', 'device_unique_senders', 'transaction_type_enc']


---

## 3. Feature Engineering

### 3.1 Temporal Features

In [109]:
# TODO: Create time-based features (hour, day of week, is_weekend, is_night, etc.)

def add_temporal_features(df):
    """Extract time-based fraud signals from the timestamp column."""
    df = df.copy()
    df['timestamp']   = pd.to_datetime(df['timestamp'])
    df['hour']        = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek    # 0=Monday, 6=Sunday
    df['month']       = df['timestamp'].dt.month
    df['is_night']    = df['hour'].between(0, 5).astype(int)
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
    return df

print("Temporal features: hour, day_of_week, month, is_night, is_weekend")

Temporal features: hour, day_of_week, month, is_night, is_weekend


### 3.2 Transaction Velocity Features

In [110]:
# TODO: Create features capturing transaction frequency/velocity
# --- 3.2 Transaction Velocity Features ---
# Captures how active a sender is on the platform. High velocity
# relative to platform norms can indicate automated fraud scripts
# or account takeover with rapid fund extraction.

def add_velocity_features(df):
    """Sender-level transaction frequency and receiver diversity."""
    df = df.copy()

    sender_velocity = df.groupby('sender_id').agg(
        sender_total_tx      =('transaction_id', 'count'),
        sender_unique_recv   =('receiver_id', 'nunique'),
    )

    df = df.merge(sender_velocity, on='sender_id', how='left')
    return df

print("Velocity features: sender_total_tx, sender_unique_recv")

Velocity features: sender_total_tx, sender_unique_recv


### 3.3 Amount-Based Features

In [111]:
# TODO: Create amount-related features (deviation from average, ratio features, etc.)
# --- 3.3 Amount-Based Features ---
# Raw amount has skewness of 9.11 — unusable directly. Log-transform
# normalises the distribution. EDA showed 94% fraud rate in the 50K+
# band, justifying is_high_value. amount_vs_sender_avg captures whether
# a transaction is anomalously large for that specific sender.

def add_amount_features(df):
    """Log-transform, high-value flag, and sender deviation ratio."""
    df = df.copy()
    df['log_amount']    = np.log1p(df['amount'])
    df['is_high_value'] = (df['amount'] > 10000).astype(int)

    sender_avg = df.groupby('sender_id')['amount'].mean().rename('sender_avg_amount')
    df = df.merge(sender_avg, on='sender_id', how='left')
    df['amount_vs_sender_avg'] = df['amount'] / (df['sender_avg_amount'] + 1)
    df.drop(columns=['sender_avg_amount'], inplace=True)

    return df

print("Amount features: log_amount, is_high_value, amount_vs_sender_avg")


Amount features: log_amount, is_high_value, amount_vs_sender_avg


### 3.4 Behavioral Features

In [ ]:
# TODO: Create features capturing user behavior patterns
# --- 3.4 Behavioral Features ---
# EDA showed 84.9% of fraud transactions come from multi-device senders
# vs 57.4% for legitimate. is_device_switch flags when a sender uses
# a device other than their most frequent one — a strong account
# takeover signal. balance_drain_rate captures the proportion of
# the sender's balance consumed — fraudsters drain accounts more completely.

def add_behavioural_features(df):
    """Device switching, balance draining, and sender device breadth."""
    df = df.copy()

    # Sender unique device count
    sender_devices = df.groupby('sender_id')['device_id'].nunique().rename('sender_unique_devices')
    df = df.merge(sender_devices, on='sender_id', how='left')

    # Primary device per sender (most frequently used)
    primary_device = (
        df.groupby(['sender_id', 'device_id'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .drop_duplicates('sender_id')
        .rename(columns={'device_id': 'primary_device'})[['sender_id', 'primary_device']]
    )
    df = df.merge(primary_device, on='sender_id', how='left')
    df['is_device_switch'] = (df['device_id'] != df['primary_device']).astype(int)
    df.drop(columns=['primary_device'], inplace=True)

    # Balance drain
    df['balance_drain_rate'] = (df['amount'] / (df['sender_balance_before'] + 1)).clip(0, 1)

    return df

print("Behavioural features: sender_unique_devices, is_device_switch, balance_drain_rate")


### 3.5 Device & Location Features

In [112]:
# TODO: Create device and location-based features
# --- 3.5 Device & Location Features ---
# 29.3% of fraud transactions originate outside Kenya vs 0.0% legitimate.
# 35.9% of fraud beyond 500km from Nairobi vs 0.0% legitimate.
# These were the strongest separators found in EDA.
# Raw lat/lon are dropped after these features are created —
# the engineered versions carry all the geographic signal.

def add_geographic_features(df):
    """Distance from Nairobi and Kenya boundary flag."""
    df = df.copy()

    NAIROBI_LAT, NAIROBI_LON = -1.2921, 36.8219

    def haversine_km(lat, lon):
        R    = 6371
        lat1 = np.radians(NAIROBI_LAT)
        lat2 = np.radians(lat)
        dlat = np.radians(lat - NAIROBI_LAT)
        dlon = np.radians(lon - NAIROBI_LON)
        a    = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        return R * 2 * np.arcsin(np.sqrt(a))

    df['dist_from_nairobi'] = haversine_km(df['location_lat'], df['location_lon'])
    df['is_outside_kenya']  = (
        (df['location_lat'] < -5) | (df['location_lat'] > 5) |
        (df['location_lon'] < 34) | (df['location_lon'] > 42)
    ).astype(int)

    # Device sharing level
    device_senders = df.groupby('device_id')['sender_id'].nunique().rename('device_unique_senders')
    df = df.merge(device_senders, on='device_id', how='left')

    return df

print("Device & location features: dist_from_nairobi, is_outside_kenya, device_unique_senders")

Device & location features: dist_from_nairobi, is_outside_kenya, device_unique_senders


In [113]:
# --- 3.6 Balance Consistency Features ---
# Sender and receiver balance changes should be consistent with the
# transaction amount. Discrepancies may indicate data manipulation
# or system-level fraud.

def add_balance_features(df):
    """Balance change and consistency check features."""
    df = df.copy()
    df['sender_balance_change']   = df['sender_balance_before'] - df['sender_balance_after']
    df['receiver_balance_change'] = df['receiver_balance_after'] - df['receiver_balance_before']
    df['balance_discrepancy']     = (
        (df['sender_balance_change'] - df['amount']).abs() > 1
    ).astype(int)
    return df

print("Balance features: sender_balance_change, receiver_balance_change, balance_discrepancy")

Balance features: sender_balance_change, receiver_balance_change, balance_discrepancy


In [114]:
# --- 3.7 Encode Categoricals ---
# Explicit mapping ensures consistent encoding across train and test.
# EDA showed transaction_type alone is a weak predictor (3.7–4.5%
# fraud rate range) but it interacts with amount and time features.

def encode_categoricals(df):
    """Label encode transaction_type with a fixed mapping."""
    df = df.copy()
    type_mapping = {
        'send_money': 0,
        'pay_bill'  : 1,
        'buy_goods' : 2,
        'withdraw'  : 3,
        'deposit'   : 4
    }
    df['transaction_type_enc'] = df['transaction_type'].map(type_mapping)
    return df

print("Categorical features: transaction_type_enc")

Categorical features: transaction_type_enc


In [115]:
# --- 3.8 Master engineer_features() function ---

def engineer_features(df):
    """
    Full feature engineering pipeline — callable on both train and test sets.
    Order matters: velocity and amount aggregates must run before
    behavioural features that depend on sender_avg_amount.
    """
    df = add_temporal_features(df)
    df = add_velocity_features(df)
    df = add_amount_features(df)
    df = add_behavioural_features(df)
    df = add_geographic_features(df)
    df = add_balance_features(df)
    df = encode_categoricals(df)

    drop_cols = [
        'timestamp', 'sender_id', 'receiver_id', 'device_id',
        'transaction_type',
    ]
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    return df


# Apply to clean training data
df_featured = engineer_features(df_clean)

print(f"✅ Feature engineering complete")
print(f"   Shape : {df_featured.shape}")
print(f"\nFinal features ({df_featured.shape[1]-1}):")
print([c for c in df_featured.columns if c != 'is_fraud'])

✅ Feature engineering complete
   Shape : (50000, 32)

Final features (31):
['transaction_id', 'amount', 'sender_balance_before', 'sender_balance_after', 'receiver_balance_before', 'receiver_balance_after', 'location_lat', 'location_lon', 'hour', 'day_of_week', 'month', 'is_night', 'is_weekend', 'sender_total_tx_x', 'sender_unique_recv_x', 'log_amount', 'is_high_value', 'amount_vs_sender_avg', 'sender_total_tx_y', 'sender_avg_amount', 'sender_unique_devices', 'sender_unique_recv_y', 'is_device_switch', 'device_unique_senders_x', 'dist_from_nairobi', 'is_outside_kenya', 'device_unique_senders_y', 'sender_balance_change', 'receiver_balance_change', 'balance_discrepancy', 'transaction_type_enc']


In [117]:
df_featured.head(10)

,transaction_id,amount,sender_balance_before,sender_balance_after,receiver_balance_before,receiver_balance_after,location_lat,location_lon,is_fraud,hour,day_of_week,month,is_night,is_weekend,sender_total_tx_x,sender_unique_recv_x,log_amount,is_high_value,amount_vs_sender_avg,sender_total_tx_y,sender_avg_amount,sender_unique_devices,sender_unique_recv_y,is_device_switch,device_unique_senders_x,dist_from_nairobi,is_outside_kenya,device_unique_senders_y,sender_balance_change,receiver_balance_change,balance_discrepancy,transaction_type_enc
0,6EB4D46BA1B3D436,701.740000,93725.40,93023.66,155223.20,155924.94,-0.479970,39.534387,0,0,0,1,1,0,9,9,6.554987,0,0.054445,9,12887.951111,3,9,0,2,314.806524,0,2,701.74,701.74,0,0
1,B6483765E2F120B3,262.180000,168781.46,168519.28,71813.67,72075.85,-1.025984,37.128477,0,1,0,1,1,0,12,12,5.572838,0,0.090853,12,2884.765287,3,12,0,2,45.135904,0,2,262.18,262.18,0,1
2,1FFF0622A1441845,446.420000,140544.59,140098.17,40409.07,40855.49,-0.570625,34.854586,0,1,0,1,1,0,6,6,6.103498,0,0.025359,6,17602.676667,1,6,0,1,232.973319,0,1,446.42,446.42,0,0
3,4EE0C335368E4144,69999.000000,162797.10,92798.10,124725.77,194724.77,4.514332,32.957873,1,1,0,1,1,0,5,5,11.156251,1,4.346015,5,16105.480000,2,5,1,2,775.346433,1,2,69999.00,69999.00,0,1
4,D769B217438B8593,75000.000000,122467.79,47467.79,132308.22,207308.22,-2.692356,37.730200,1,2,0,1,1,0,11,11,11.225257,1,7.555904,11,9925.012485,3,11,1,3,185.555233,0,3,75000.00,75000.00,0,0
5,71E680082779E166,1983.060000,40586.40,38603.34,126164.78,128147.84,0.479754,35.276640,0,2,0,1,1,0,9,9,7.592901,0,1.736879,9,1140.737242,4,9,0,2,261.414158,0,2,1983.06,1983.06,0,0
6,260508735EFF1B52,187.960000,20237.86,20049.90,38281.62,38469.58,-0.211970,34.734479,0,2,0,1,1,0,5,5,5.241535,0,0.085943,5,2186.038000,2,5,0,1,261.322921,0,1,187.96,187.96,0,2
7,366039223B5056F0,35926.500700,139749.42,103822.92,81498.14,117424.64,-0.154283,34.728174,1,3,0,1,1,0,9,9,10.489258,1,6.087324,9,5900.854522,4,9,1,2,264.949151,0,2,35926.50,35926.50,0,0
8,C1AA1BA6158DBB5A,527.430000,26993.13,26465.70,91667.21,92194.64,-0.329649,36.059911,0,3,0,1,1,0,7,7,6.269910,0,0.302831,7,1740.667143,1,7,0,3,136.494171,0,3,527.43,527.43,0,2
9,8AFF359986635CED,562.079495,119718.73,119156.65,103558.59,104120.67,-1.503282,37.315814,1,4,0,1,1,0,4,4,6.333421,0,1.131810,4,495.619874,3,4,1,1,59.715243,0,1,562.08,562.08,0,0


### Feature Engineering Summary

*TODO: Document your engineered features and rationale*

| Feature | Description | Rationale |
|---------|-------------|-----------||
| | | |
| | | |

In [116]:
# --- 3.9 Feature Engineering Summary ---

summary = {
    'Temporal': [
        ('hour',            'Hour of day (0–23)',                        'Peak fraud at 03h (39.6% rate)'),
        ('day_of_week',     'Day of week (0=Mon)',                       'Monday highest fraud day-of-week'),
        ('month',           'Month of year',                             'Seasonal pattern capture'),
        ('is_night',        'Binary: 00h–05h',                          'Late-night = low monitoring window'),
        ('is_weekend',      'Binary: Sat/Sun',                          'Lower volume, elevated fraud risk'),
    ],
    'Velocity': [
        ('sender_total_tx', 'Total transactions by sender',             'High velocity = possible fraud script'),
        ('sender_unique_recv','Unique receivers per sender',            'Many receivers = money mule pattern'),
    ],
    'Amount': [
        ('log_amount',           'log(1 + amount)',                     'Corrects skewness of 9.11'),
        ('is_high_value',        'Binary: amount > KES 10,000',        '94% fraud rate in 50K+ band'),
        ('amount_vs_sender_avg', 'Amount / sender historical mean',    'Flags anomalously large transactions'),
    ],
    'Behavioural': [
        ('sender_unique_devices','Unique devices per sender',          '84.9% fraud from multi-device senders'),
        ('is_device_switch',     'Binary: not using primary device',   'Account takeover signal'),
        ('balance_drain_rate',   'Amount / sender_balance_before',     'Fraudsters drain accounts more fully'),
    ],
    'Geographic': [
        ('dist_from_nairobi',  'Haversine distance from Nairobi (km)', '35.9% fraud >500km vs 0.0% legitimate'),
        ('is_outside_kenya',   'Binary: outside lat/lon Kenya bounds', '29.3% fraud outside Kenya vs 0.0%'),
        ('device_unique_senders','Unique senders per device',         'Shared device = fraud ring indicator'),
    ],
    'Balance': [
        ('sender_balance_change',   'Before minus after (sender)',     'Consistency check on transaction'),
        ('receiver_balance_change', 'After minus before (receiver)',   'Receiver-side anomaly detection'),
        ('balance_discrepancy',     'Binary: change ≠ amount',        'System manipulation flag'),
    ],
    'Categorical': [
        ('transaction_type_enc', 'Encoded transaction type (0–4)',     'Type interacts with amount + time'),
    ],
}

print(f"{'Feature':<30} {'Description':<40} {'Rationale'}")
print("-" * 110)
for group, features in summary.items():
    print(f"\n[{group}]")
    for feat, desc, rationale in features:
        print(f"  {feat:<28} {desc:<40} {rationale}")

Feature                        Description                              Rationale
--------------------------------------------------------------------------------------------------------------

[Temporal]
  hour                         Hour of day (0–23)                       Peak fraud at 03h (39.6% rate)
  day_of_week                  Day of week (0=Mon)                      Monday highest fraud day-of-week
  month                        Month of year                            Seasonal pattern capture
  is_night                     Binary: 00h–05h                          Late-night = low monitoring window
  is_weekend                   Binary: Sat/Sun                          Lower volume, elevated fraud risk

[Velocity]
  sender_total_tx              Total transactions by sender             High velocity = possible fraud script
  sender_unique_recv           Unique receivers per sender              Many receivers = money mule pattern

[Amount]
  log_amount                   log(1 

---

## 4. Data Preprocessing

In [122]:
# TODO: Prepare final feature set
# =============================================================================
# STEP 4: DATA PREPROCESSING
# =============================================================================

import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

os.makedirs('models', exist_ok=True)

# --- 4.1 Prepare Final Feature Set ---
# Drop raw columns superseded by engineered versions.
# transaction_id is kept in df_featured as an index reference
# but excluded from X.

drop_before_model = [
    'transaction_id',  # identifier — no predictive value
    'amount',          # superseded by log_amount
    'location_lat',    # superseded by dist_from_nairobi, is_outside_kenya
    'location_lon',    # superseded by dist_from_nairobi, is_outside_kenya
]

df_model = df_featured.drop(
    columns=[c for c in drop_before_model if c in df_featured.columns]
)

FEATURE_COLS = [c for c in df_model.columns if c != 'is_fraud']
TARGET_COL   = 'is_fraud'

X = df_model[FEATURE_COLS]
y = df_model[TARGET_COL]

print(f"✅ Final feature set prepared")
print(f"   Features : {len(FEATURE_COLS)}")
print(f"   Samples  : {len(X):,}")
print(f"   Fraud    : {y.sum():,} ({y.mean()*100:.1f}%)")

ModuleNotFoundError: No module named 'multiprocessing'

In [123]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib # This should work now!

# 1. Define features (excluding IDs and raw targets)
# Update this list based on what you actually named your engineered columns
features = [col for col in df_featured.columns if col not in ['transaction_id', 'is_fraud', 'timestamp', 'amount']]
X = df_featured[features]
y = df_featured['is_fraud']

# 2. Train/Test Split (80/20) with Stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Scaling (Essential for many models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Handle Imbalance with SMOTE
# This creates synthetic fraud examples so the model learns the pattern better
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print("✅ Environment Fixed & Data Preprocessed")
print(f"Training set size: {X_train_res.shape[0]} (Balanced)")
print(f"Test set size: {X_test.shape[0]} (Original Ratio)")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# TODO: Train/test split


In [ ]:
# TODO: Handle class imbalance (SMOTE, class weights, etc.)


In [ ]:
# TODO: Feature scaling if needed


---

## 5. Model Development

### 5.1 Baseline Model

In [ ]:
# TODO: Train a baseline model (e.g., Logistic Regression)


### 5.2 Model Comparison

In [ ]:
# TODO: Train and compare multiple models
# Consider: Random Forest, XGBoost, LightGBM, CatBoost, etc.


### 5.3 Hyperparameter Tuning

In [ ]:
# TODO: Tune the best performing model


### 5.4 Final Model Evaluation

In [ ]:
# TODO: Evaluate the final model on test set
# Show: Confusion matrix, Classification report, ROC curve, PR curve


### Model Performance Summary

*TODO: Document your model's performance*

| Metric | Value |
|--------|-------|
| AUC-ROC | |
| Precision | |
| Recall | |
| F1-Score | |

---

## 6. Model Interpretation

In [ ]:
# TODO: Feature importance analysis


In [ ]:
# TODO: SHAP values (optional but recommended)


---

## 7. Conclusions & Next Steps

### Summary

*TODO: Write your summary*

### Key Findings

*TODO: List your key findings*

1. 
2. 
3. 

### Production Considerations

*TODO: Discuss what would be needed for production deployment*

### Limitations & Future Improvements

*TODO: Discuss limitations and potential improvements*

In [ ]:
# TODO: Save the final model


---

## 8. Test Set Predictions ⭐ REQUIRED

**IMPORTANT:** You MUST generate predictions for the test set and include them in your submission.

The test data is in `data/test_transactions.json` and does NOT contain the `is_fraud` column.

In [ ]:
# Load the test data
import json

with open('../data/test_transactions.json', 'r') as f:
    test_data = json.load(f)

test_df = pd.DataFrame(test_data)
print(f"Test transactions: {len(test_df)}")
test_df.head()

In [ ]:
# TODO: Apply the same feature engineering to test data
# IMPORTANT: Use the same transformations you applied to training data

# Example:
# test_df = engineer_features(test_df)  # Your feature engineering function
# test_features = test_df[feature_columns]  # Same columns as training


In [ ]:
# TODO: Generate predictions using your trained model

# Example:
# predictions = model.predict(test_features)


In [ ]:
# Create the submission file
# File must have columns: transaction_id, predicted_fraud

submission_df = pd.DataFrame({
    'transaction_id': test_df['transaction_id'],
    'predicted_fraud': predictions  # Your predictions (0 or 1)
})

# Verify format
print(f"Submission shape: {submission_df.shape}")
print(f"Unique predictions: {submission_df['predicted_fraud'].unique()}")
print(f"\nPrediction distribution:")
print(submission_df['predicted_fraud'].value_counts())

submission_df.head()

In [ ]:
# Save predictions to CSV
submission_df.to_csv('test_predictions.csv', index=False)
print("✅ Predictions saved to test_predictions.csv")
print("\n⚠️ Remember to include this file in your submission!")